# Analysis of temporal analysis
This code was written by Sai and Danyka Byrnes

In [317]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from config import *

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

#### Answer the following questions based on the timeseries data:
1) Percentage of Data missing between the start and end date.
  
2) The length between start and end datetime (hour) 

3) longest continous recored between the start and end date.  

In [318]:
# List of possible parameters
parameters = [
    "WTemp_C", "SpC_uScm", "DO_mgL", "pH",
    "Turb_FNU", "Turb_NTU", "NO3_mgNL",
    "fDOM_QSU", "fDOM_RFU", "DOC_mgL",
    "PO4_mgL", "Chla_ugL", "Chla_RFU",
    "PC_ugL", "PC_RFU"
]

# Store all results
results = []

# Loop through each CSV
for file in os.listdir(water_quality_filepath):
    if not file.endswith(".csv"):
        continue

    file_path = os.path.join(water_quality_filepath, file)
    
    # Extract STREAM_ID from filename (remove .csv)
    stream_id = os.path.splitext(file)[0]

    df = pd.read_csv(file_path, low_memory=False) # mixed data types in df, maybe worth double checking

    if "DateTime" not in df.columns:
        continue

    # Convert DateTime
    df["DateTime"] = pd.to_datetime(df["DateTime"], errors="coerce")
    df = df.dropna(subset=["DateTime"])
    df = df.sort_values("DateTime")

    # Create full hourly range
    start = df["DateTime"].min()
    end = df["DateTime"].max()
    full_range = pd.date_range(start=start, end=end, freq="h")

    # Reindex to full hourly timeline
    df = df.set_index("DateTime").reindex(full_range)

    # Total expected length in hours
    total_hours = len(full_range)

    station_result = {"STREAM_ID": stream_id}

    for param in parameters:
        if param in df.columns:

            series = df[param]

            # 1 Percentage Missing
            missing_count = series.isna().sum()
            percent_missing = (missing_count / total_hours) * 100

            # 2 Length between start and end
            length_hours = total_hours

            # 3 Longest continuous record (non-missing streak)
            not_na = series.notna().astype(int)
            groups = (not_na.diff() != 0).cumsum()
            streak_lengths = not_na.groupby(groups).sum()
            longest_streak = streak_lengths.max() if len(streak_lengths) > 0 else 0

            station_result[f"{param}_pct_missing"] = round(percent_missing, 2)
            station_result[f"{param}_length_hr"] = length_hours
            station_result[f"{param}_longest_hr"] = int(longest_streak)

        else:
            # If parameter not present
            station_result[f"{param}_pct_missing"] = np.nan
            station_result[f"{param}_length_hr"] = np.nan
            station_result[f"{param}_longest_hr"] = np.nan

    results.append(station_result)

# Convert to dataframe
temporal_summary = pd.DataFrame(results)

print("Processing complete.")

Processing complete.


In [319]:
# Merging the temporal data derived by Sai
# Identify ID/metadata columns (keep these as-is)
id_cols = ['STREAM_ID']

# Reshape
rows = []
for temp_param in temporal_params:
    temp = temporal_summary[id_cols].copy()
    temp['parameter'] = temp_param
    temp['pct_missing'] = temporal_summary[f'{temp_param}_pct_missing']
    temp['length_hr'] = temporal_summary[f'{temp_param}_length_hr']
    temp['longest_hr'] = temporal_summary[f'{temp_param}_longest_hr']
    rows.append(temp)

temporal_summary_long = (
    pd.concat(rows, ignore_index=True)
    .query('parameter in @temporal_params')
    .dropna(how='all', subset=['pct_missing', 'length_hr', 'longest_hr'])
)
temporal_summary_long.head()
temporal_summary_long.to_csv(OUTPUT_filepath+'temporal_statistics.csv', index=False)

In [320]:
# Processing metadata, isolating specific parameters
md_wide = pd.read_csv(metadata_filepath+"metadata.csv", dtype = {'sourceID': str})
print(f"Full dataset size: {md_wide.shape[0]}")
params = ['SpC_uScm','DO_mgL','Turb_FNU']

for param in params:
    md_wide[param] = md_wide['WQ_parameters'].str.contains(param)

# Removing the stations without data
md_wide = md_wide[(md_wide['SpC_uScm']) | (md_wide['DO_mgL']) | (md_wide['Turb_FNU'])]
print(f"Filtered dataset size: {md_wide.shape[0]}")

Full dataset size: 870
Filtered dataset size: 556


In [321]:
md_wide.head()

,STREAM_ID,sourceID,source,site name,latitude_wgs84,longitude_wgs84,drainagearea_sqkm,state_name,time_zone,WQ_parameters,SpC_uScm,DO_mgL,Turb_FNU
0,STREAM-gauge-1723,03007800,USGS,"Allegheny River at Port Allegany, PA",41.818676,-78.292791,642.317520,Pennsylvania,Eastern,"WTemp_C,SpC_uScm,DO_mgL,pH",True,True,False
2,STREAM-gauge-1724,03012545,USGS,"Allegheny River below Kinzua Dam at Big Bend, PA",41.838672,-79.004925,5646.178200,Pennsylvania,Eastern,"WTemp_C,DO_mgL",False,True,False
3,STREAM-gauge-1725,03012550,USGS,"Allegheny River at Kinzua Dam, PA",41.841449,-79.011985,5646.178200,Pennsylvania,Eastern,"WTemp_C,DO_mgL",False,True,False
4,STREAM-gauge-1726,03016000,USGS,"Allegheny River at West Hickory, PA",41.570895,-79.407824,9479.363400,Pennsylvania,Eastern,"WTemp_C,SpC_uScm",True,False,False
11,STREAM-gauge-1733,03027500,USGS,"EB Clarion River at EB Clarion River Dam, PA",41.553118,-78.596135,189.587268,Pennsylvania,Eastern,"WTemp_C,SpC_uScm,DO_mgL,pH,Turb_FNU",True,True,True
